<a href="https://colab.research.google.com/github/FabioFloris02/NLP2026_Floris_Sonzini_Parenti_Sarra_Rossi/blob/main/Model_testing.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Cosa possiamo testare e far vedere?**
## **prompting e generazione delle risposte:**
- Tasso di risposta corretta con le diverse strategie di similarità accoppiate con un modello Llama che risponde generando una risposta generica (non direttamente l'opzione corretta)
- Tasso di risposta corretta con il modello che genera direttamente l'opzione che gli diamo
- Dalla consegna: What prompt (zero, few shot, etc.) gives the best results?
- Dalla consegna: How sensitive are different LLMs to different prompts?
- Dalla consegna: What types of questions do the models tend to struggle on?
- Dalla consegna: Is the model often overconfident in its answers and does that affect its performance?
- Dalla consegna: Should the same prompt be used for all questions or made more specific as the questions get harder?
## **models:**
- Dalla consegna: Are some LLMs better than others at answering the questions?
- Dalla consegna: Are bigger models better than smaller models?
- Dalla consegna: Are the models performing as well as a human on this task?
- Dalla consegna: Are certain models better at certain topics than others? (Es: controllare se ci sono info sui dataset sui quali sono stati allenati)
- Dalla consegna: Are “thinking” models better at answering questions than “non-thinking” ones?
## **improvements:**
- Dalla consegna: Is there a way to fine-tune a model to improve its performance? If so, what data could you use to train the model?


## **Similarity da provare:**
- Regex extraction
- TF-IDF + Cosine Similarity
- Sentence-BERT (sBERT) Semantic Similarity (quella testata è biencoder dell'esercitazione Session7? è implementata in maniera un po' diversa ma l'idea sembra simile)
- Sentence-BERT con CrossEncoder (è un po' più lenta ma dovrebbe performare meglio. essendoci solo 5 frasi da confrontare non dovrebbe essere un problema. Vedi session7 per implementarla)

# **Imports and installs**

In [2]:
!pip install -q -U transformers
!pip install -q scikit-learn
!pip install -q sentence-transformers

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.6/10.6 MB 114.9 MB/s eta 0:00:00


In [3]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, pipeline
from google.colab import userdata
from huggingface_hub import login
import re
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np
from sentence_transformers import SentenceTransformer
import numpy as np
from typing import Callable
import os
import pandas as pd
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, pipeline
import sys
import time
from typing import Callable
from sentence_transformers import CrossEncoder


# **Setup HuggingFace and Game APIs**

## **HuggingFace**

In [4]:
HF_TOKEN = userdata.get('HF_TOKEN')
login(HF_TOKEN)

## **Game APIs**

Let's import the client API folder from our GitHib repository

In [5]:
repo_url = "https://github.com/FabioFloris02/NLP2026_Floris_Sonzini_Parenti_Sarra_Rossi.git"
repo_name = "NLP2026_Floris_Sonzini_Parenti_Sarra_Rossi"

if os.path.exists("../"+repo_name):
    print("Repository already present, update...")
    !git pull
else:
    print("Repository clone...")
    !git clone {repo_url}
    %cd {repo_name}

sys.path.append('/content/NLP2026_Floris_Sonzini_Parenti_Sarra_Rossi/NLP_assignment_api_client')

from millionaire_client import MillionaireClient, AuthenticationError, GameError

Repository clone...
Cloning into 'NLP2026_Floris_Sonzini_Parenti_Sarra_Rossi'...
remote: Enumerating objects: 144, done.
remote: Counting objects: 100% (144/144), done.
remote: Compressing objects: 100% (133/133), done.
Receiving objects: 100% (144/144), 161.30 KiB | 4.24 MiB/s, done.
remote: Total 144 (delta 65), reused 15 (delta 4), pack-reused 0 (from 0)
Resolving deltas: 100% (65/65), done.
/content/NLP2026_Floris_Sonzini_Parenti_Sarra_Rossi


Let's check if we are correctly logged in.

In [6]:
API_URL  = 'http://131.175.15.22:51111/'
USERNAME = 'GliEmbeddingRuspanti'
PASSWORD = 'GliEmbeddingRuspanti'

client = MillionaireClient(API_URL)
try:
    user = client.login(USERNAME, PASSWORD)
    print(f'Logged in as: {user.username} (role: {user.role})')
except AuthenticationError as e:
    print(f'Login failed: {e}')

Logged in as: GliEmbeddingRuspanti (role: student)


# **Model classes**

Here we build a model class so that we can easily define a model and implement as a method how the effective answering logic is implemented.

For instance we can generate a full response through a text-generation and then compute the answer of the model through similarity.

In [7]:
class Model():
    """
    Il modello generates the output.
    answer_fn decide how to get the final option.
    """

    def __init__(self, name: str, answer_fn: Callable[[str, dict], str]):
        self.name = name
        self.answer_fn = answer_fn

    def generate(self, question: str, system_prompt: str = "") -> str:
        pass

    def answer(self, question: str, options: dict, system_prompt: str = "") -> str:
        """Generates and process the answer through answer_fn."""
        raw_output = self.generate(question, system_prompt)
        print(f"MODEL OUTPUT ----> CAZZO FUNZIONA + {raw_output}")
        return self.answer_fn(raw_output, options)

    def __repr__(self):
        return f"{self.__class__.__name__}(name={self.name!r}, answer_fn={self.answer_fn.__name__!r})"

class HFPipelineModel(Model):
    DEFAULT_GEN_ARGS = {
        "max_new_tokens": 600,
        "return_full_text": False,
        "temperature": 0.5,
        "do_sample": True,
    }

    def __init__(
        self,
        name: str,
        model_name: str,
        answer_fn: Callable[[str, dict], str],
        hf_token: str | None = None,
        device_map: str = "cuda",
        gen_args: dict | None = None,
        cache_dir: str | None = None,
    ):
        super().__init__(name, answer_fn)
        self.gen_args = {**self.DEFAULT_GEN_ARGS, **(gen_args or {})}

        model = AutoModelForCausalLM.from_pretrained(
            model_name, device_map=device_map, torch_dtype="auto",
            trust_remote_code=True, token=hf_token, cache_dir=cache_dir,
        )
        tokenizer = AutoTokenizer.from_pretrained(model_name, token=hf_token, cache_dir=cache_dir)
        self._pipe = pipeline("text-generation", model=model, tokenizer=tokenizer)

    def generate(self, question: str, system_prompt: str = "") -> str:
        messages = [
            {"role": "system", "content": system_prompt},
            {"role": "user",   "content": question},
        ]
        output = self._pipe(messages, **self.gen_args)
        return output[0]["generated_text"]

# **Answers logic implementation through functions**

Here we'll implement the logic behind the true answer we'll give to the game.

## **Regex direct extraction**

In [8]:
# Strategy 1: Regex-based direct extraction

def extract_by_regex(model_output: str):
    text = model_output.strip()

    m = re.search(r'(?:answer(?:\s+is)?|option|choice|select|pick|correct)\s*[:\-]?\s*\*{0,2}([ABCD])\*{0,2}', text, re.IGNORECASE)
    if m: return m.group(1).upper()

    m = re.search(r'\b([ABCD])[\)\.:](?:\s|$)', text, re.IGNORECASE)
    if m: return m.group(1).upper()
    # lone capital letter (models often deliberate, then conclude with the letter)
    letters = re.findall(r'(?<![a-zA-Z])([ABCD])(?![a-zA-Z])', text)
    if letters:
        return letters[-1].upper()
    return None

# Let's set up a small test to see if it works
test_cases = [
    ('The answer is A, because Napoleon was a political figure.', 'A'),
    ('Napoleon was a ruler. I think the answer is B.', 'B'),
    ('C) Un politico', 'C'),
    ('Napoleon era un generale. La risposta corretta e D.', 'D'),
    ('He was born in Corsica and rose to become emperor', None),
]
print('Regex extraction tests:')
for text, expected in test_cases:
    result = extract_by_regex(text)
    status = 'PASS' if result == expected else 'FAIL'
    print(f'  [{status}] Input: {repr(text[:55]):57s} -> Got: {result}, Expected: {expected}')

Regex extraction tests:
  [PASS] Input: 'The answer is A, because Napoleon was a political figur' -> Got: A, Expected: A
  [PASS] Input: 'Napoleon was a ruler. I think the answer is B.'          -> Got: B, Expected: B
  [PASS] Input: 'C) Un politico'                                          -> Got: C, Expected: C
  [PASS] Input: 'Napoleon era un generale. La risposta corretta e D.'     -> Got: D, Expected: D
  [PASS] Input: 'He was born in Corsica and rose to become emperor'       -> Got: None, Expected: None


## **TF-IDF + cosine similarity**

In [9]:
# Strategy 2: TF-IDF + Cosine Similarity (Vector Space Model)

def pick_by_tfidf(model_output: str, options: dict):

    labels = list(options.keys())
    texts = [model_output] + [options[l] for l in labels]

    # Fit TF-IDF on all texts together so the vocabulary is shared
    vectorizer = TfidfVectorizer()
    tfidf_matrix = vectorizer.fit_transform(texts)

    query_vec = tfidf_matrix[0]
    option_vecs = tfidf_matrix[1:]

    sims = cosine_similarity(query_vec, option_vecs)[0]
    scores = {labels[i]: float(sims[i]) for i in range(len(labels))}
    best = max(scores, key=scores.get)
    return best

# Let's set up a small test to see if it works
options_test = {
    'A': 'Un politico',
    'B': 'un personaggio televisivo',
    'C': 'qualcosa di assurdo',
    'D': 'Un calciatore'
}

#model_response = 'Napoleone era un grande leader politico e militare, imperatore dei francesi.'
#best_tfidf, scores_tfidf = pick_by_tfidf(model_response, options_test)
#print('TF-IDF cosine similarity scores:')
#for label, score in sorted(scores_tfidf.items(), key=lambda x: -x[1]):
 #   marker = ' <-- SELECTED' if label == best_tfidf else ''
  #  print(f'  {label} ({options_test[label]!r:30s}): {score:.4f}{marker}')
#print(f'\nPicked option: {best_tfidf}')

## **sBERT: A semantic similarity approach**

In [10]:
# Strategy 3: Sentence-BERT (sBERT) Semantic Similarity

sbert_model = SentenceTransformer('paraphrase-multilingual-MiniLM-L12-v2')

def pick_by_sbert(model_output: str, options: dict):

    labels = list(options.keys())
    all_texts = [model_output] + [options[l] for l in labels]

    embeddings = sbert_model.encode(all_texts, convert_to_numpy=True, normalize_embeddings=True)

    query_emb = embeddings[0]
    option_embs = embeddings[1:]

    sims = option_embs @ query_emb
    scores = {labels[i]: float(sims[i]) for i in range(len(labels))}
    best = max(scores, key=scores.get)
    return best
# Let's set up a small test to see if it works
options_test = {
    'A': 'Un politico',
    'B': 'un personaggio televisivo',
    'C': 'qualcosa di assurdo',
    'D': 'Un calciatore'
}

model_response = 'Napoleone era un grande leader politico e militare, imperatore dei francesi.'

#best_sbert, scores_sbert = pick_by_sbert(model_response, options_test)
#print('sBERT semantic similarity scores:')
#for label, score in sorted(scores_sbert.items(), key=lambda x: -x[1]):
 #   marker = ' <-- SELECTED' if label == best_sbert else ''
#    print(f'  {label} ({options_test[label]!r:30s}): {score:.4f}{marker}')
#print(f'\nPicked option: {best_sbert}')

modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/122 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/645 [00:00<?, ?B/s]

...

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/526 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.08M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

## **Biencoder (da finire)**

In [11]:
multilingual_model = SentenceTransformer('paraphrase-multilingual-mpnet-base-v2')
def pick_by_sbert_biencoder(model_output: str, options: dict):
  labels = list(options.keys())
  all_texts = [model_output] + [options[l] for l in labels]

  embeddings = sbert_model.encode(all_texts, convert_to_numpy=True, normalize_embeddings=True)

  query_emb = embeddings[0]
  option_embs = embeddings[1:]


modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/122 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/723 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/402 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

## **Cross encoder**

In [12]:
modelCrossencoder = CrossEncoder('cross-encoder/stsb-distilroberta-base')
def pick_by_crossencoder(model_output: str, options: dict):
    all_options = list(options.values())
    roberta_inputs = [[model_output, opt] for opt in all_options]
    scores = modelCrossencoder.predict(roberta_inputs)
    return np.argmax(scores)


config.json:   0%|          | 0.00/607 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/328M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json: 0.00B [00:00, ?B/s]

# **Models**

Let's define a set of models we want to use.

Specifically we'll embed models in a Model class object. We'll specify the aswering logic through one of the previous defined classes.

In the end we'll make a full set so that we can easily test all the models and have benchmarks.

In [13]:
'''
models = {'model name': [Model, system_prompt]}
'''

"\nmodels = {'model name': [Model, system_prompt]}\n"

In [14]:
# 1. Only TF-IDF
llama_tf_idf = HFPipelineModel(
    name="llama-sbert",
    model_name="meta-llama/Llama-3.2-1B-Instruct",
    answer_fn=lambda raw_output, opts: pick_by_tfidf(raw_output, opts),
    hf_token=HF_TOKEN,
    cache_dir="./models_cache",
)

# 2. Only sBERT
llama_sbert = HFPipelineModel(
    name="llama-regex-sbert",
    model_name="meta-llama/Llama-3.2-1B-Instruct",
    answer_fn=lambda raw_output, opts: pick_by_sbert(raw_output, opts),
    hf_token=HF_TOKEN,
    cache_dir="./models_cache",
)

# 3. Crossencoder
llama_crossencoder = HFPipelineModel(
    name="llama-roberta",
    model_name="meta-llama/Llama-3.2-1B-Instruct",
    answer_fn=lambda raw_output, opts: pick_by_crossencoder(raw_output, opts),
    hf_token=HF_TOKEN,
    cache_dir="./models_cache",
)





config.json:   0%|          | 0.00/877 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/2.47G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/146 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/189 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/296 [00:00<?, ?B/s]

Loading weights:   0%|          | 0/146 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/146 [00:00<?, ?it/s]

In [15]:
models = {
    '''"llama_sbert_genericPrompt": {
        "model": llama_sbert,
        "system_prompt": "You are a quiz game expert. Answer in the most exhaustive manner."
    },

    "llama_tfidf_genericPrompt": {
        "model": llama_tf_idf,
        "system_prompt": "You are a quiz game expert. Answer in the most exhaustive manner."
    },

    "llama_crossencoder_genericPrompt": {
        "model": llama_crossencoder,
        "system_prompt": "You are a quiz game expert. Answer in the most exhaustive manner."
    },'''

    "llama_sbert_robustPrompt": {
        "model": llama_sbert,
        "system_prompt": """
            You are a highly skilled competitive quiz player. Your primary objective is to maximize answer accuracy and provide the most factually correct response possible for every question.

            Behavior rules:
            - Always try to determine the correct answer using reasoning, world knowledge, context clues, and inference.
            - If the answer is uncertain, provide the most probable answer rather than refusing to answer.
            - Avoid random guessing when possible; make educated inferences instead.
            - Prefer concise, direct answers over long explanations.
            - Do not roleplay, joke, or add unnecessary commentary.
            - Do not intentionally hedge unless uncertainty is genuinely high.
            - Use careful internal reasoning before answering.
            - If multiple answers seem possible, choose the one most likely to be accepted in a standard quiz context.
            - Prioritize commonly accepted canonical answers.
            - Be robust to ambiguous wording and infer likely intent.
            - Optimize for correctness over creativity or personality.

            Output rules:
            - Respond with only the final answer.
            - Do not explain your reasoning unless explicitly requested.
            - Keep answers short and precise.
        """
    },

    "llama_tfidf_robustPrompt": {
        "model": llama_tf_idf,
        "system_prompt": """
            You are a highly skilled competitive quiz player. Your primary objective is to maximize answer accuracy and provide the most factually correct response possible for every question.

            Behavior rules:
            - Always try to determine the correct answer using reasoning, world knowledge, context clues, and inference.
            - If the answer is uncertain, provide the most probable answer rather than refusing to answer.
            - Avoid random guessing when possible; make educated inferences instead.
            - Prefer concise, direct answers over long explanations.
            - Do not roleplay, joke, or add unnecessary commentary.
            - Do not intentionally hedge unless uncertainty is genuinely high.
            - Use careful internal reasoning before answering.
            - If multiple answers seem possible, choose the one most likely to be accepted in a standard quiz context.
            - Prioritize commonly accepted canonical answers.
            - Be robust to ambiguous wording and infer likely intent.
            - Optimize for correctness over creativity or personality.

            Output rules:
            - Respond with only the final answer.
            - Do not explain your reasoning unless explicitly requested.
            - Keep answers short and precise.
        """     },

    "llama_crossencoder_robustPrompt": {
        "model": llama_crossencoder,
        "system_prompt": """
            You are a highly skilled competitive quiz player. Your primary objective is to maximize answer accuracy and provide the most factually correct response possible for every question.

            Behavior rules:
            - Always try to determine the correct answer using reasoning, world knowledge, context clues, and inference.
            - If the answer is uncertain, provide the most probable answer rather than refusing to answer.
            - Avoid random guessing when possible; make educated inferences instead.
            - Prefer concise, direct answers over long explanations.
            - Do not roleplay, joke, or add unnecessary commentary.
            - Do not intentionally hedge unless uncertainty is genuinely high.
            - Use careful internal reasoning before answering.
            - If multiple answers seem possible, choose the one most likely to be accepted in a standard quiz context.
            - Prioritize commonly accepted canonical answers.
            - Be robust to ambiguous wording and infer likely intent.
            - Optimize for correctness over creativity or personality.

            Output rules:
            - Respond with only the final answer.
            - Do not explain your reasoning unless explicitly requested.
            - Keep answers short and precise.
        """     }
}

# **The Game**

In [16]:
def play_game(game, model, sys_prompt):
  log = []
  while game.in_progress:
      question = game.current_question
      if not question:
          print("No question available. Game may have ended.")
          break

      print(f"\n--- Level {game.current_level} ---")
      print(f"Q: {question.text}")
      print()

      for opt in question.options:
          print(f"  [{opt.id}] {opt.text}")

      time_left = game.time_remaining
      if time_left:
          print(f"\nTime remaining: {time_left:.1f}s")

      options = {f"{opt.id}": opt.text for opt in question.options}

      t0 = time.time()
      answer_input = model.answer(question.text, options, sys_prompt)
      inference_time = time.time() - t0
      print(f"Model answer: {answer_input}")
      answer_id = int(answer_input)

      choosen_answer = question.options[answer_id]

      result = game.answer(answer_id)

      if result.correct:
          print(" CORRECT!")
          if result.game_over:
              print(f"\n CONGRATULATIONS! You completed the game!")
              print(f" Final earnings: ${result.earned_amount:,.2f}")
          else:
              print(f" Earned so far: ${result.earned_amount:,.2f}")
      elif result.timed_out:
        print("TIMED OUT!")
        print(f"\n Game Over!")
        print(f" Final earnings: ${result.earned_amount:,.2f}")
      elif not result.correct:
          print(" WRONG ANSWER!")
          print(f"\n Game Over!")
          print(f" Final earnings: ${result.earned_amount:,.2f}")

      # Log the outcome
      entry = {
          'level'           : game.current_level,
          'question'        : question.text,
          'options'         : question.options,
          'chosen_option'   : choosen_answer.text,
          'correct'         : result.correct,
          'timed_out'       : result.timed_out,
          'inference_time'  : round(inference_time, 2),
      }
      log.append(entry)

  summary = {
        'model'           : model.name,
        'final_level'     : game.current_level,
        'earned_amount'   : game.earned_amount,
        'num_questions'   : len(log),
        'num_correct'     : sum(1 for e in log if e['correct']),
        'num_timed_out'   : sum(1 for e in log if e['timed_out']),
        'avg_inference_s' : round(sum(e['inference_time'] for e in log) / max(len(log), 1), 2),
        'log'             : log,
    }

  print("\n=== Game Summary ===")
  print(f"Reached Level: {game.current_level}")
  print(f"Total Earnings: ${game.earned_amount:,.2f}")

  return summary

In [17]:
results = {}

for model_name, config in models.items():
    print(f"\n########## MODEL: {model_name} ##########")

    model = config["model"]
    system_prompt = config["system_prompt"]

    model_results = []

    for comp_id in [0, 1, 2, 3]:
        print(f"\n--- Competition {comp_id} ---")

        game = client.game.start(competition_id=comp_id)

        summary = play_game(game, model, system_prompt)

        model_results.append(summary)

    results[model_name] = model_results


########## MODEL: "llama_sbert_genericPrompt": {
        "model": llama_sbert,
        "system_prompt": "You are a quiz game expert. Answer in the most exhaustive manner."
    },

    "llama_tfidf_genericPrompt": {
        "model": llama_tf_idf,
        "system_prompt": "You are a quiz game expert. Answer in the most exhaustive manner."
    },

    "llama_crossencoder_genericPrompt": {
        "model": llama_crossencoder,
        "system_prompt": "You are a quiz game expert. Answer in the most exhaustive manner."
    },llama_sbert_robustPrompt ##########

--- Competition 0 ---


[transformers] Passing `generation_config` together with generation-related arguments=({'do_sample', 'max_new_tokens', 'temperature'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
[transformers] Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.



--- Level 1 ---
Q: What is the fundamental principle of Paul McCartney's approach to songwriting and music creation, as described in the article?

  [0] Versatility and exploration across various genres
  [1] Limited involvement in the recording process
  [2] Focusing solely on rock and roll
  [3] Strict adherence to classical music forms

Time remaining: 29.9s


[transformers] Both `max_new_tokens` (=600) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer TokenizersBackend. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.


MODEL OUTPUT ----> CAZZO FUNZIONA + I'm unable to find any information on a specific article that describes Paul McCartney's approach to songwriting and music creation.
Model answer: 1
 WRONG ANSWER!

 Game Over!
 Final earnings: $0.00

=== Game Summary ===
Reached Level: 1
Total Earnings: $0.00

--- Competition 1 ---


...

[transformers] Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
[transformers] Both `max_new_tokens` (=600) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


MODEL OUTPUT ----> CAZZO FUNZIONA + Heat.
Model answer: 0
 CORRECT!
 Earned so far: $1,000.00

--- Level 6 ---
Q: Decomposers are important to the flow of energy in an ecosystem because they

  [0] break organic material into parts that can be reused.
  [1] are at the top of every food chain.
  [2] make up a large portion of the top layer of the soil.
  [3] eat things other organisms do not eat.

Time remaining: 29.9s
MODEL OUTPUT ----> CAZZO FUNZIONA + Microorganisms that break down dead organic matter, such as bacteria and fungi, are crucial to the flow of energy in an ecosystem.
Model answer: 3
 WRONG ANSWER!

 Game Over!
 Final earnings: $1,000.00

=== Game Summary ===
Reached Level: 6
Total Earnings: $1,000.00

--- Competition 3 ---


[transformers] Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
[transformers] Both `max_new_tokens` (=600) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



--- Level 1 ---
Q: If Wayne considers the letter “Y” to be a vowel but Kristen does not, thinking that there are only 5 vowels, by what percent is the probability that a randomly selected letter out of the 26 letter alphabet will be a vowel greater in Wayne’s opinion than in Kristen’s opinion?

  [0] 5%
  [1] 6%
  [2] 30%
  [3] 20%

Time remaining: 29.9s
MODEL OUTPUT ----> CAZZO FUNZIONA + To find the probability that a randomly selected letter is a vowel, we first need to determine the number of vowels and consonants. 

There are 5 vowels (A, E, I, O, U) and 21 consonants (all letters except A, E, I, O, U).

The probability of selecting a vowel is 5/26, and the probability of selecting a consonant is 21/26.

To find the percent increase in probability, we need to find the difference in the probabilities and divide by the original probability:

(5/26 - 21/26) / (5/26) * 100%
= (-16/26) / (5/26) * 100%
= -16/5 * 100%
= -3200%

However, this is the percentage decrease in probability. Si

MillionaireError: Could not connect to server at http://131.175.15.22:51111: ('Connection aborted.', RemoteDisconnected('Remote end closed connection without response'))

##**Print of results**

In [ ]:
def print_results(results):
    for model_name, competitions in results.items():

        print("\n" + "=" * 80)
        print(f"MODELLO: {model_name}")
        print("=" * 80)

        for i, summary in enumerate(competitions):

            print(f"\n🏁 Competition {i}")
            print("-" * 60)

            print(f"Model name        : {summary['model']}")
            print(f"Final level       : {summary['final_level']}")
            print(f"Earned amount     : €{summary['earned_amount']}")
            print(f"Questions         : {summary['num_questions']}")
            print(f"Correct answers   : {summary['num_correct']}")
            print(f"Timed out         : {summary['num_timed_out']}")
            print(f"Avg inference     : {summary['avg_inference_s']} s")

            accuracy = (
                summary['num_correct'] / summary['num_questions'] * 100
                if summary['num_questions'] > 0 else 0
            )

            print(f"Accuracy          : {accuracy:.1f}%")

            print("\n📋 Question Log")
            print("-" * 60)

            for q_idx, entry in enumerate(summary['log'], start=1):

                status = "✅" if entry['correct'] else "❌"

                if entry.get('timed_out'):
                    status = "⏰"

                print(
                    f"{q_idx:02d}. "
                    f"{status} "
                    f"Time: {entry['inference_time']:.2f}s"
                )

        print("\n")


# stampa finale
print_results(results)